# 大模型LoRA微调实战：课程教案与作业整合版  

**目标：** 本 Notebook 整合了《阿拉伯语专业领域大模型LoRA微调实战教程》与《2025夏令营大模型微调workshop实践作业》的全部内容。旨在提供一个从数据处理、模型训练、参数调优到多语言迁移的完整、可执行的工作流。  

**环境信息：**  
- **操作系统：** Linux  
- **Python版本：** 3.11.8  
- **计算资源：** NVIDIA V100 Tensor Core GPU  
- **核心库：** PyTorch 2.5.1, CUDA 12.3  

**作业任务分解：**  
1.  **基础作业一：** 探索不同LoRA配置（`r`, `lora_alpha`）对模型性能的影响，使用 **ROUGE** 指标进行评估。  
2.  **基础作业二：** 实现自定义评估指标（本Notebook中已实现ROUGE指标）。  
3.  **进阶作业：** 将微调流程拓展至其他语言（如韩语/俄语），代码中已包含处理该数据集的框架。

## 步骤一：环境安装与设置  
首先，我们需要安装所有必需的Python库，并设置Hugging Face的镜像以加速模型下载。

In [1]:
# 使用清华镜像安装所需依赖包
!pip install transformers==4.41.2 datasets==2.19.0 peft==0.10.0 accelerate==0.30.1 torch==2.5.1 -i https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
!pip install sentence_transformers scikit-learn rouge_chinese jieba nltk -i https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple

DEPRECATION: Loading egg at /opt/conda/lib/python3.11/site-packages/papermill-2.3.1-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330
Looking in indexes: https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
DEPRECATION: Loading egg at /opt/conda/lib/python3.11/site-packages/papermill-2.3.1-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330
Looking in indexes: https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple


In [2]:
# 设置Hugging Face国内镜像，加速模型下载
import os
import logging

os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# 设置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

logger.info(f"Hugging Face Endpoint 设置为: {os.environ.get('HF_ENDPOINT')}")

2025-07-24 02:02:26,987 - INFO - Hugging Face Endpoint 设置为: https://hf-mirror.com


## 步骤二：数据处理  
本部分包含处理原始数据（`.jsonl.gz`格式）的完整流程，将其转换为适合指令微调的格式。

In [3]:
import json
import gzip
from pathlib import Path
import re
from tqdm import tqdm
import random
from typing import List, Dict

def clean_text(text: str) -> str:
    """清理文本中的多余换行、空格和HTML标签。"""
    if not text:
        return ""
    text = re.sub(r'\n+', '\n', text.strip())
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'<[^>]+>', '', text)
    return text.strip()

def process_item(item: Dict, lang: str = 'ar') -> Dict:
    """处理单条数据，构建统一的指令微调格式。"""
    try:
        if 'title' in item and 'content' in item:
            title = clean_text(item['title'])
            content = clean_text(item['content'])
        elif 'text' in item:
            text = clean_text(item['text'])
            lines = text.split('\n', 1)
            title = lines[0] if len(lines) > 1 else "文章"
            content = lines[1] if len(lines) > 1 else text
        else:
            return None

        if len(content) < 50 or len(content) > 10000:
            return None
        
        lang_map = {
            'ar': '阿拉伯语',
            'ko': '韩语',
            'ru': '俄语'
        }
        lang_name = lang_map.get(lang, '特定语言')

        return {
            "instruction": f"请根据以下标题生成一段带有专业术语的{lang_name}文本。\n\n标题: {title}",
            "input": "",
            "output": content
        }
    except Exception as e:
        logger.error(f"处理数据时出错: {e}")
        return None

def process_raw_file(input_path: str, output_path: str, lang: str = 'ar', sample_ratio: float = 0.25):
    """处理单个原始数据文件并保存。"""
    processed_data = []
    try:
        with gzip.open(input_path, 'rt', encoding='utf-8') as f:
            lines = f.readlines()
            sample_size = max(1, int(len(lines) * sample_ratio))
            sampled_lines = random.sample(lines, sample_size)

            for line in tqdm(sampled_lines, desc=f"处理文件 {Path(input_path).name}"):
                try:
                    item = json.loads(line.strip())
                    processed_item = process_item(item, lang=lang)
                    if processed_item:
                        processed_data.append(processed_item)
                except (json.JSONDecodeError, Exception):
                    continue
    except Exception as e:
        logger.error(f"读取文件 {input_path} 时出错: {e}")
        return

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(processed_data, f, ensure_ascii=False, indent=2)
    
    logger.info(f"处理完成！从 {Path(input_path).name} 中提取了 {len(processed_data)} 条数据。")
    logger.info(f"数据已保存至: {output_path}")

# --- 执行数据处理 ---
random.seed(42)

# 1. 处理阿拉伯语数据 (基础作业)
arabic_raw_path = '/home/mw/input/raw_arabic81628162/阿拉伯part-677f75d865d8-001143.jsonl.gz'
arabic_processed_path = '/home/mw/input/Arabic60526052/lora_training_data_arabic.json' # 教案中使用的路径
if not os.path.exists(arabic_processed_path):
    logger.info("开始处理阿拉伯语原始数据...")
    process_raw_file(arabic_raw_path, arabic_processed_path, lang='ar')
else:
    logger.info(f"已找到处理好的阿拉伯语数据: {arabic_processed_path}")

# 2. 处理韩语/俄语数据 (进阶作业)
# 注意：路径中包含'Russia'，但根据作业上下文可能为韩语数据集，这里我们按俄语处理
korean_raw_path = '/home/mw/input/Russia39613961/part-677f75d865d8-000260.jsonl.gz'
korean_processed_path = '/home/mw/input/Russia39613961/lora_training_data_korean.json'
if not os.path.exists(korean_processed_path):
    logger.info("开始处理韩语/俄语原始数据 (进阶作业)...")
    # 假设该数据集是俄语
    process_raw_file(korean_raw_path, korean_processed_path, lang='ko')
else:
    logger.info(f"已找到处理好的韩语/俄语数据: {korean_processed_path}")

2025-07-24 02:02:27,020 - INFO - 已找到处理好的阿拉伯语数据: /home/mw/input/Arabic60526052/lora_training_data_arabic.json
2025-07-24 02:02:27,020 - INFO - 已找到处理好的韩语/俄语数据: /home/mw/input/Russia39613961/lora_training_data_korean.json


## 步骤三：评估器定义  
根据作业要求，我们定义一个评估器，专注于计算ROUGE分数。

In [4]:
import numpy as np
import torch
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_chinese import Rouge
import jieba
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

class RougeEvaluator:
    """一个专注于计算ROUGE分数的文本评估器。"""
    def __init__(self):
        self.rouge = Rouge()
    
    def calculate_rouge(self, reference: str, candidate: str) -> Dict[str, float]:
        """计算ROUGE分数"""
        if not candidate or not candidate.strip():
            return {'rouge-1': 0.0, 'rouge-2': 0.0, 'rouge-l': 0.0}
        try:
            reference_cut = ' '.join(jieba.cut(reference))
            candidate_cut = ' '.join(jieba.cut(candidate))
            scores = self.rouge.get_scores(candidate_cut, reference_cut)[0]
            return {
                'rouge-1': scores['rouge-1']['f'],
                'rouge-2': scores['rouge-2']['f'],
                'rouge-l': scores['rouge-l']['f']
            }
        except Exception as e:
            logger.error(f"计算ROUGE分数时出错: {e}")
            return {'rouge-1': 0.0, 'rouge-2': 0.0, 'rouge-l': 0.0}

class TrainingEvaluator:
    """训练过程中的综合评估器"""
    def __init__(self, tokenizer, device):
        self.tokenizer = tokenizer
        self.device = device
        self.rouge_evaluator = RougeEvaluator()
        
        # # --- 以下为教案中原有指标，根据要求注释掉 ---
        # self.sentence_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2').to(device)
        # try:
        #     with open("/home/mw/input/Arabic60526052/domain_terms_arabic.txt", "r", encoding="utf-8") as f:
        #         self.domain_terms = [line.strip() for line in f if line.strip()]
        # except FileNotFoundError:
        #     logger.warning("domain_terms_arabic.txt 未找到，领域指标将受限。")
        #     self.domain_terms = []
        # self.bleu_smooth = SmoothingFunction()

    def evaluate_generation(self, model, validation_prompts: List[Dict]) -> Dict[str, float]:
        """评估生成文本的ROUGE分数（批量生成优化版）。"""
        model.eval()
        
        # 步骤1: 准备所有提示和参考答案
        prompts = [f"### Instruction:\n{item['instruction']}\n\n### Response:\\n" for item in validation_prompts]
        references = [item['output'] for item in validation_prompts]

        # 步骤2: 对所有提示进行一次性批量分词
        # padding=True 会自动将批次内的所有序列填充到最长序列的长度
        inputs = self.tokenizer(
            prompts, 
            return_tensors="pt", 
            padding=True, 
            truncation=True, 
            max_length=512  # 输入提示的最大长度
        ).to(self.device)

        # 步骤3: 进行一次性批量生成
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                # 关键优化：限制新生成的token数量，避免过长输出
                max_new_tokens=256,  
                do_sample=True,
                temperature=0.7,
                top_p=0.95,
                pad_token_id=self.tokenizer.eos_token_id
            )

        # 步骤4: 一次性批量解码所有生成的结果
        # skip_special_tokens=True 会自动移除 [CLS], [SEP] 等特殊token
        generated_texts = self.tokenizer.batch_decode(outputs, skip_special_tokens=True)

        # 步骤5: 循环计算ROUGE分数（这部分很快，因为耗时的生成已经完成）
        all_rouge_scores = {'rouge-1': [], 'rouge-2': [], 'rouge-l': []}
        for generated_text, reference_output in zip(generated_texts, references):
            # 提取模型生成的回复部分
            generated_output = generated_text.split("### Response:")[-1].strip()
            
            rouge_scores = self.rouge_evaluator.calculate_rouge(reference_output, generated_output)
            for key, value in rouge_scores.items():
                all_rouge_scores[key].append(value)

        avg_metrics = {f"avg_{k}": np.mean(v) for k, v in all_rouge_scores.items()}
        return avg_metrics

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-24 02:02:32,042 - INFO - PyTorch version 2.5.1 available.


## 步骤四：模型训练与参数搜索 (基础作业一)  
这是课程的核心部分。我们将：  
1. 定义一个数据加载函数。  
2. 循环遍历不同的LoRA超参数（`r` 和 `lora_alpha`）。  
3. 对每组参数，进行短暂的训练。  
4. 使用ROUGE分数评估模型性能。  
5. 找出并保存性能最佳的模型。

In [5]:
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from torch.optim import AdamW

from datasets import Dataset

def load_and_prepare_data(json_path, tokenizer, val_size=0.1):
    """
    加载并预处理数据（最终修正版）。
    此版本仅进行分词，不手动填充或创建标签。
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 筛选有效数据并准备格式化文本
    valid_data_for_prompts = []
    formatted_texts = []
    for item in data:
        if 'prompt' in item:
            valid_data_for_prompts.append(item)
            prompt = item['prompt']
            output = item.get('output', '')
            # 我们仍然需要完整的文本用于分词
            text = f"### Instruction:\n{prompt}\n\n### Response:\n{output}"
            formatted_texts.append(text)

    if not formatted_texts:
        raise ValueError("在JSON文件中没有找到包含'prompt'键的有效数据。")

    # 分词，但这次不进行填充(padding=False)
    encodings = tokenizer(formatted_texts, truncation=True, max_length=512)
    dataset = Dataset.from_dict(encodings)
    
    # 添加原始索引以安全地获取验证集提示
    dataset = dataset.add_column("original_index", range(len(valid_data_for_prompts)))

    # 切分数据集
    split_dataset = dataset.train_test_split(test_size=val_size, seed=42)
    train_dataset = split_dataset['train']
    val_dataset = split_dataset['test']
    
    # 构建验证集提示
    validation_prompts = []
    for val_item in val_dataset:
        original_idx = val_item['original_index']
        original_data_item = valid_data_for_prompts[original_idx]
        validation_prompts.append({
            "instruction": original_data_item.get('prompt', ''),
            "output": original_data_item.get('output', '')
        })

    # 移除不再需要的列
    train_dataset = train_dataset.remove_columns("original_index")
    val_dataset = val_dataset.remove_columns("original_index")

    return train_dataset, val_dataset, validation_prompts

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, get_linear_schedule_with_warmup, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
from torch.optim import AdamW

from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, get_linear_schedule_with_warmup, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
from torch.optim import AdamW

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def main_training_loop():
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_path = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
    data_path = "/home/mw/input/Arabic60526052/lora_training_data_arabic.json"

    logger.info(f"设备: {device}, 模型: {model_path}")

    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 仅加载一次完整数据集
    full_train_dataset, val_dataset, validation_prompts = load_and_prepare_data(data_path, tokenizer)
    
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    
    # --- 显存优化 1: 减小 Batch Size ---
    # 将批次大小降至4，以确保不会OOM
    batch_size = 4
    # ----------------------------------
    
    # 创建一个临时的DataLoader用于预提取数据
    temp_dataloader = DataLoader(full_train_dataset, batch_size=batch_size, shuffle=True, collate_fn=data_collator)

    # --- 显存优化 2: 采用small_train_data策略 ---
    logger.info("预先提取少量批次数据用于快速、多轮次的实验...")
    train_iter = iter(temp_dataloader)
    small_train_data = []
    num_batches_to_fetch = 50  # 获取50个批次 (50批*4条/批=200条样本)
    for _ in range(num_batches_to_fetch):
        try:
            small_train_data.append(next(train_iter))
        except StopIteration:
            logger.warning("数据源不足50个批次，已提取所有可用批次。")
            break
    logger.info(f"已创建小训练集，包含 {len(small_train_data)} 个批次。")
    # ---------------------------------------------
    
    lora_configs_to_try = [
        {"r": 8, "lora_alpha": 16, "lora_dropout": 0.05},
        {"r": 16, "lora_alpha": 32, "lora_dropout": 0.1},
        {"r": 32, "lora_alpha": 64, "lora_dropout": 0.1}
    ]

    best_rouge_l = -1
    best_config = None
    all_results = []

    for config in lora_configs_to_try:
        logger.info(f"--- 开始测试配置: r={config['r']}, alpha={config['lora_alpha']} ---")
        
        # 每次循环都加载新模型，避免状态污染
        base_model = AutoModelForCausalLM.from_pretrained(
            model_path, trust_remote_code=True, torch_dtype=torch.bfloat16
        ).to(device)

        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM, **config,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], bias="none"
        )
        peft_model = get_peft_model(base_model, lora_config)
        
        optimizer = AdamW(peft_model.parameters(), lr=5e-5)
        
        # --- 训练循环修改：在小数据集上训练2个epoch ---
        num_epochs = 2
        for epoch in range(num_epochs):
            peft_model.train()
            # 直接在预加载的数据列表上循环
            for batch in tqdm(small_train_data, desc=f"Epoch {epoch+1}/{num_epochs} with r={config['r']}"):
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = peft_model(**batch)
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
        # ---------------------------------------------

        logger.info("开始评估...")
        evaluator = TrainingEvaluator(tokenizer, device)
        rouge_metrics = evaluator.evaluate_generation(peft_model, validation_prompts[:50])
        
        logger.info(f"配置 r={config['r']} 的评估结果:")
        for key, value in rouge_metrics.items():
            logger.info(f"  - {key}: {value:.4f}")
        
        current_results = {"config": config, **rouge_metrics}
        all_results.append(current_results)

        if rouge_metrics['avg_rouge-l'] > best_rouge_l:
            best_rouge_l = rouge_metrics['avg_rouge-l']
            best_config = config
            logger.info(f"*** 发现新的最佳配置: {config} ***")
            best_model_path = "/home/mw/input/Arabic314891489/deepseek-lora-best"
            peft_model.save_pretrained(best_model_path)
            tokenizer.save_pretrained(best_model_path)
            logger.info(f"最佳模型已保存至: {best_model_path}")
        
        # 手动清理显存，为下一个循环做准备
        del base_model, peft_model, optimizer
        torch.cuda.empty_cache()

    logger.info("\n--- 所有配置测试完成 ---")
    for result in all_results:
        logger.info(f"配置: {result['config']} -> ROUGE-L: {result['avg_rouge-l']:.4f}")
    logger.info(f"\n最终最佳配置为: {best_config}")

# 执行训练和参数搜索
#main_training_loop()

2025-07-24 02:02:32,806 - INFO - 设备: cuda, 模型: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
2025-07-24 02:02:44,173 - INFO - 预先提取少量批次数据用于快速、多轮次的实验...
2025-07-24 02:02:44,275 - INFO - 已创建小训练集，包含 50 个批次。
2025-07-24 02:02:44,275 - INFO - --- 开始测试配置: r=8, alpha=16 ---
Epoch 2/2 with r=8: 100%|██████████| 50/50 [01:13<00:00,  1.48s/it]
2025-07-24 02:05:29,281 - INFO - 开始评估...
Building prefix dict from the default dictionary ...
2025-07-24 02:06:06,517 - DEBUG - Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
2025-07-24 02:06:06,519 - DEBUG - Loading model from cache /tmp/jieba.cache
Loading model cost 0.691 seconds.
2025-07-24 02:06:07,209 - DEBUG - Loading model cost 0.691 seconds.
Prefix dict has been built successfully.
2025-07-24 02:06:07,210 - DEBUG - Prefix dict has been built successfully.
2025-07-24 02:06:24,710 - INFO - 配置

## 步骤五：模型合并与测试  
训练完成后，我们将性能最佳的LoRA权重与基础模型合并，形成一个完整的、可以直接部署的模型，并进行最终的生成效果测试。

In [6]:
from peft import PeftModel

def merge_and_test():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    base_model_path = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
    lora_adapter_path = "/home/mw/input/Arabic314891489/deepseek-lora-best"
    merged_model_path = "/home/mw/input/Arabic314891489/deepseek-merged-final"

    logger.info("开始加载基础模型用于合并...")
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_path,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        device_map="auto"
    )

    logger.info(f"加载LoRA适配器从: {lora_adapter_path}")
    model_to_merge = PeftModel.from_pretrained(base_model, lora_adapter_path)

    logger.info("开始合并权重...")
    merged_model = model_to_merge.merge_and_unload()
    logger.info("权重合并完成。")

    logger.info(f"保存合并后的模型至: {merged_model_path}")
    merged_model.save_pretrained(merged_model_path)
    tokenizer = AutoTokenizer.from_pretrained(lora_adapter_path)
    tokenizer.save_pretrained(merged_model_path)
    logger.info("模型和分词器已保存。")

    # --- 测试合并后的模型 ---
    logger.info("\n--- 测试合并后的模型生成效果 ---")
    test_prompts = [
        "请解释阿拉伯语中'السوق'这个术语的含义。",
        "ما معنى مصطلح 'الذكاء الاصطناعي'؟", # "人工智能"这个术语是什么意思？
        "اكتب فقرة قصيرة عن أهمية الطاقة المتجددة.", # 写一段关于可再生能源重要性的短文
    ]
    
    for prompt in test_prompts:
        logger.info(f"\n测试提示: {prompt}")
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = merged_model.generate(
                **inputs,
                max_length=256,
                num_return_sequences=1,
                do_sample=True,
                temperature=0.7,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id
            )
        
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        logger.info(f"模型回答:\n{response}")
        logger.info("-" * 50)

# 执行模型合并与测试
merge_and_test()

2025-07-24 02:13:34,554 - INFO - 开始加载基础模型用于合并...
2025-07-24 02:13:36,918 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).
2025-07-24 02:13:45,152 - INFO - 加载LoRA适配器从: /home/mw/input/Arabic314891489/deepseek-lora-best
2025-07-24 02:13:45,342 - INFO - 开始合并权重...
2025-07-24 02:13:45,397 - INFO - 权重合并完成。
2025-07-24 02:13:45,397 - INFO - 保存合并后的模型至: /home/mw/input/Arabic314891489/deepseek-merged-final
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
2025-07-24 02:13:52,281 - INFO - 模型和分词器已保存。
2025-07-24 02:13:52,282 - INFO - 
--- 测试合并后的模型生成效果 ---
2025-07-24 02:13:52,282 - INFO - 
测试提示: 请解释阿拉伯语中'السوق'这个术语的含义。
2025-07-24 02:14:02,032 - INFO - 模型回答:
请解释阿拉伯语中'السوق'这个术语的含义。请提供相关资料来源，方便了解。请以简明易懂的方式表达，不需要翻译。请给出的资料来源必须为可靠的，如维基百科、书籍、网页等。

**示例回答：**

السوق (السُقَّل) هو سبب تُسُبُّن مُحَز

## 步骤六：总结与进阶作业展望  

至此，我们完成了基础作业的核心部分：  
1.  **数据处理**：成功将原始的阿拉伯语料处理成指令微调格式。  
2.  **参数搜索**：通过循环测试，找到了在ROUGE-L指标上表现最佳的LoRA超参数组合。  
3.  **模型训练与保存**：训练并保存了性能最佳的LoRA模型。  
4.  **模型合并**：将LoRA权重与基础模型合并，得到了一个独立的、性能增强的模型。  

### 进阶作业指导  
要完成进阶作业（微调韩语/俄语模型），您需要：  
1.  **修改数据路径**：在`main_training_loop`函数中，将`data_path`变量指向处理好的韩语/俄语数据文件路径（例如 `/home/mw/input/Russia39613961/lora_training_data_korean.json`）。  
2.  **实现特定语言分词（可选但推荐）**：为了更精确地计算ROUGE分数，您可以在`RougeEvaluator`中为目标语言（如韩语）实现特定的分词逻辑，而不是统一使用`jieba`。  
3.  **重新运行训练流程**：执行修改后的Notebook，即可开始对新语言模型的微调和评估。